# GP4 ReAct-IR Qwen2.5 QLoRA Cloud Workflow

Colab runtime must persist generated data, reports, checkpoints, adapters, and inference outputs to Google Drive only.

The notebook uses `gp4_finetune_factory_source_bundle.zip` from `$CLOUD_ROOT/bundles/` when present. Otherwise it clones the pushed `codex/gp4-react-ir-cloud-workflow` branch into the ephemeral runtime. Generated data, reports, checkpoints, adapters, inference outputs, and packages stay under `CLOUD_ROOT`.


In [1]:
import os
import shutil
import zipfile
from datetime import datetime
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
RUN_ID = os.environ.get('RUN_ID') or 'gp4-react-50k-' + datetime.utcnow().strftime('%Y%m%d-%H%M')
CLOUD_ROOT = f'/content/drive/MyDrive/gp4_finetune_factory/{RUN_ID}'
SOURCE_BUNDLE = f'{CLOUD_ROOT}/bundles/gp4_finetune_factory_source_bundle.zip'
WORK_DIR = Path('/content/gp4_finetune_factory_source')
assert SOURCE_BUNDLE.startswith('/content/drive/'), 'Source bundle must live in Google Drive.'
os.makedirs(f'{CLOUD_ROOT}/reports', exist_ok=True)
os.makedirs(f'{CLOUD_ROOT}/manifests', exist_ok=True)
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
if Path(SOURCE_BUNDLE).exists():
    with zipfile.ZipFile(SOURCE_BUNDLE) as archive:
        archive.extractall(WORK_DIR)
else:
    !git clone --branch codex/gp4-react-ir-cloud-workflow --depth 1 https://github.com/Hieu-RMX18/gp4_finetune_factory.git {WORK_DIR}
os.chdir(WORK_DIR)
SEED_PATH = str(WORK_DIR / 'data/seed/gp4_seed_starter.jsonl')
SOURCE_PLAN = str(WORK_DIR / 'docs/superpowers/plans/2026-05-16-gp4-react-ir-cloud-workflow.md')
os.environ['RUN_ID'] = RUN_ID
os.environ['CLOUD_ROOT'] = CLOUD_ROOT
os.environ['SEED_PATH'] = SEED_PATH
os.environ['SOURCE_PLAN'] = SOURCE_PLAN
print(CLOUD_ROOT)
print(WORK_DIR)


Mounted at /content/drive


/tmp/ipykernel_2724/1170480906.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_ID = os.environ.get('RUN_ID') or 'gp4-react-50k-' + datetime.utcnow().strftime('%Y%m%d-%H%M')


Cloning into '/content/gp4_finetune_factory_source'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 80 (delta 7), reused 53 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (80/80), 99.19 KiB | 1.50 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/drive/MyDrive/gp4_finetune_factory/gp4-react-50k-20260517-1635
/content/gp4_finetune_factory_source


In [ ]:
import os
from google.colab import userdata

if not os.environ.get('DEEPSEEK_API_KEY'):
    deepseek_api_key = userdata.get('DEEPSEEK_API_KEY')
    if deepseek_api_key:
        os.environ['DEEPSEEK_API_KEY'] = deepseek_api_key
assert os.environ.get('DEEPSEEK_API_KEY'), 'Set DEEPSEEK_API_KEY in the cloud runtime secrets before generation.'
os.environ['DEEPSEEK_BASE_URL'] = os.environ.get('DEEPSEEK_BASE_URL', 'https://api.deepseek.com')
os.environ['HF_HOME'] = f'{CLOUD_ROOT}/.cache/huggingface'
os.environ['TRANSFORMERS_CACHE'] = f'{CLOUD_ROOT}/.cache/huggingface/transformers'
os.environ['HF_DATASETS_CACHE'] = f'{CLOUD_ROOT}/.cache/huggingface/datasets'
os.environ['TORCH_HOME'] = f'{CLOUD_ROOT}/.cache/torch'
os.environ['XDG_CACHE_HOME'] = f'{CLOUD_ROOT}/.cache'
os.environ['WANDB_DIR'] = f'{CLOUD_ROOT}/wandb'
os.environ['TMPDIR'] = f'{CLOUD_ROOT}/tmp'
for key in ('HF_HOME', 'TRANSFORMERS_CACHE', 'HF_DATASETS_CACHE', 'TORCH_HOME', 'XDG_CACHE_HOME', 'WANDB_DIR', 'TMPDIR'):
    os.makedirs(os.environ[key], exist_ok=True)


In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip install -q unsloth datasets trl


In [ ]:
!python scripts/provider_probe.py --provider colab --cloud-root "$CLOUD_ROOT" --report "$CLOUD_ROOT/reports/platform_status_${RUN_ID}.json"


In [ ]:
!python scripts/cloud_orchestrator.py --run-id "$RUN_ID" --cloud-root "$CLOUD_ROOT" --seed "$SEED_PATH" --source-plan "$SOURCE_PLAN" --phases cloud-setup,provider-probe,contract,seed-check,generate-smoke,generate-1k,quality-gate-1k,generate-30k,quality-gate-30k,generate-50k,quality-gate-50k,dedupe,split,train,infer,eval,package


In [ ]:
!python scripts/audit_cloud_completion.py --cloud-root "$CLOUD_ROOT" --run-id "$RUN_ID" --report "$CLOUD_ROOT/reports/completion_audit_${RUN_ID}.json"


The orchestrator stops automatically unless the previous gate report has `passed=true`. Completion evidence is `$CLOUD_ROOT/reports/acceptance_gate_report_$RUN_ID.json` with `passed=true` and package metadata under the same cloud root.
